# Stage 11 — Pack and verify

**The question:** is this feature set complete, self-describing, and fast to
read?

The tissue feature set `uni2h-L1-224-t144-hsv-v1` is 10,616 small files, one per
slide, in `data/features/<version>/slides/`. This notebook packs it into three
files next to them, extends the manifest, and proves the packed set is correct:

| File | Contents |
|---|---|
| `features.npy` | every tile's features, fp16, [tiles, 1536], memory-mappable |
| `index.parquet` | one row per slide: `slide_id`, `start`, `end`, `n_tiles` |
| `tiles.parquet` | one row per tile: `slide_id`, `tile`, level-0 `x` and `y`, `tissue_frac` |
| `manifest.json` | the manifest notebook 9 wrote, plus a `packed` section |

**Why.** Training reads every slide every epoch; one memory-mapped array opens
instantly and reads only the rows it needs, where ten thousand small files mean
ten thousand file opens. And a feature set nobody checked is how slides go
missing, or silently receive each other's features, without anyone noticing.
The verification also fails unless the manifest says `masks_used: false`, so a
set built with mask-guided tiles can never pass as this one.

**How.** The functions live in `src/features.py`, so the packing can run again
for every future version and pytest can re-run the verification
(`tests/test_feature_set.py`). This notebook tries them on a scratch copy of a
few dozen slides, shows the verification failing on planted faults, then runs
them on the real set. Its saved outputs are the stage's record.

Only the tissue set is packed. The grid set was a floor for the Stage 9
comparison; nothing downstream reads it. The empty slide
`3790f55cad63053e956fb73027179707` (zero tiles, FINDINGS Stage 10) is left out
and listed in the manifest with its reason.

## Preamble

In [1]:
"""Make the project's src/ folder importable.

Locally it sits one folder up from notebooks/. On Kaggle it arrives as an
attached dataset, so every attached dataset is checked for a src/ folder, and
the notebook stops with a clear message if there is none.
"""

import glob
import os
import sys

SRC_ROOT = next(
    (
        root
        for root in [os.path.abspath("..")] + sorted(glob.glob("/kaggle/input/*"))
        if os.path.isfile(os.path.join(root, "src", "tiling.py"))
    ),
    None,
)
if SRC_ROOT is None:
    raise FileNotFoundError("src/ not found; on Kaggle, attach the project's src folder as a dataset")
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

In [2]:
"""Imports.

os, shutil, tempfile          paths, and the scratch copies the functions are tried on first.
time                          the two read speeds.
numpy                         planting faults in the array, and the read-speed test.
pandas                        the index and tile tables, and folds.csv.
torch                         reads the per-slide .pt files in the read-speed test.
tqdm                          progress over 10,616 slides.
IPython.display               shows tables.
src.features                  checking, packing, opening and verifying a feature set.
src.paths                     where the feature sets and folds.csv are, locally and on Kaggle.
"""

import os
import shutil
import tempfile
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm

from src.features import (
    FEATURES_FILE,
    INDEX_FILE,
    MANIFEST_FILE,
    PART_SUFFIX,
    SLIDES_SUBDIR,
    VerificationError,
    check_complete,
    count_tiles,
    open_packed,
    pack,
    read_manifest,
    set_dir,
    slide_file,
    verify,
    write_manifest,
)
from src.paths import DATA_DIR, find_input

In [3]:
"""Settings.

1. VERSION is the feature set this notebook packs: the tissue set from notebook 9.
2. EXCLUDED lists the slides left out of the packed set, each with its reason.
   It is written by hand on purpose: a zero-tile slide that is not listed here
   is an error to investigate, not something to drop quietly. After packing,
   the manifest carries this list.
3. The expected slides are every slide in the Stage 4 fold file. The fold file
   is read for its slide IDs only.
4. SCRATCH_DIR and SCRATCH_SLIDES set up the scratch copies the functions are
   tried on before they touch the real set. They live in the system's temporary
   folder, so it works locally and on Kaggle.
5. OVERWRITE must be set to True to pack the real set again (for example to
   record a clean commit in the manifest). Left False, packing a set that is
   already packed stops with an error instead of replacing it.
"""

VERSION = "uni2h-L1-224-t144-hsv-v1"
EXCLUDED = {
    "3790f55cad63053e956fb73027179707": "empty file: every pixel white at every level, zero tiles (FINDINGS, Stage 10)",
}

folds = pd.read_csv(find_input("folds.csv", local_dir=DATA_DIR))
expected_ids = sorted(folds["slide_id"])

SCRATCH_DIR = os.path.join(tempfile.gettempdir(), "stage11_scratch")
SCRATCH_SLIDES = 40
OVERWRITE = False

print(
    f"{VERSION}: {len(expected_ids)} expected slides, {len(EXCLUDED)} excluded, "
    f"{len(expected_ids) - len(EXCLUDED)} to pack"
)
print(f"real set: {set_dir(VERSION)}")
print(f"scratch copies: {SCRATCH_DIR}")

folds.csv: /home/aztrek/projects/Prostate-Cancer-Grading/data/folds.csv
uni2h-L1-224-t144-hsv-v1: 10616 expected slides, 1 excluded, 10615 to pack
real set: /home/aztrek/projects/Prostate-Cancer-Grading/data/features/uni2h-L1-224-t144-hsv-v1
scratch copies: /tmp/stage11_scratch


## The functions

They live in `src/features.py`, one copy only, and are imported above. Every
function that touches files takes the version string and a `features_dir` that
defaults to `FEATURES_DIR`, so the same code runs on the scratch copies and on
the real set:

- `set_dir`, `slide_file`: where a set and one slide's file are;
- `file_sha256`, `code_commit`, `read_manifest`, `write_manifest` (atomic:
  written under a `.part` name, then renamed);
- `check_complete`: the slides folder holds exactly the expected slides, with
  no leftovers, and the manifest names the version and says
  `masks_used: false`;
- `read_slide`: the one place a per-slide file is read, with its format
  checked;
- `count_tiles`;
- `pack`: pre-allocate, fill one slide at a time, rename the three files only
  when all are complete, then extend the manifest;
- `open_packed`: the array memory-mapped read-only, plus the index, the tile
  table and the manifest;
- `verify`: nine checks, every failure collected, one `VerificationError`
  naming them all.

## A scratch copy

A few dozen slides' files, including the empty one so the exclusion path runs,
copied with the manifest into a temporary features folder. Packing and
verifying it shows what the functions produce before they touch the real set:
the summary, the index, and the first rows of the tile table.

In [4]:
"""Pack and verify a scratch copy.

1. Pick the scratch slides by rule: evenly spaced through the sorted expected
   slides, plus the excluded ones, so the exclusion path runs.
2. make_scratch(name) copies those slides' .pt files and the manifest into a
   fresh features folder SCRATCH_DIR/name/ and returns it. With packed_from,
   it copies a whole packed scratch set instead. Every fault below gets its
   own copy, so no fault leaks into the next.
3. Pack the clean copy and verify it; show the summary, the report, the index
   and the first rows of the tile table.
"""

step = len(expected_ids) // SCRATCH_SLIDES
scratch_ids = sorted(set(expected_ids[::step][: SCRATCH_SLIDES - len(EXCLUDED)]) | set(EXCLUDED))
if os.path.isdir(SCRATCH_DIR):
    shutil.rmtree(SCRATCH_DIR)


def make_scratch(name, packed_from=None):
    features_dir = os.path.join(SCRATCH_DIR, name)
    if packed_from:
        shutil.copytree(set_dir(VERSION, packed_from), set_dir(VERSION, features_dir))
        return features_dir
    os.makedirs(os.path.join(set_dir(VERSION, features_dir), SLIDES_SUBDIR))
    for slide_id in scratch_ids:
        shutil.copy2(slide_file(VERSION, slide_id), slide_file(VERSION, slide_id, features_dir))
    shutil.copy2(os.path.join(set_dir(VERSION), MANIFEST_FILE), os.path.join(set_dir(VERSION, features_dir), MANIFEST_FILE))
    return features_dir


clean = make_scratch("clean")
summary = pack(VERSION, scratch_ids, EXCLUDED, clean)
print(f"{len(scratch_ids)} scratch slides; packed {summary['slides']} slides, {summary['tiles']} tiles, "
      f"excluded {list(summary['excluded'])}, in {summary['seconds']:.1f} s")
print("\n".join(verify(VERSION, scratch_ids, clean)))
scratch = open_packed(VERSION, clean)
print(f"features: {type(scratch.features).__name__}, {scratch.features.dtype}, {scratch.features.shape}")
display(scratch.index.head())
display(scratch.tiles.head())
print(scratch.manifest["packed"])

40 scratch slides; packed 39 slides, 2535 tiles, excluded ['3790f55cad63053e956fb73027179707'], in 0.0 s
ok    slides: 39 packed + 1 excluded = 40 expected
ok    no empty: smallest slide has 29 tiles; 1 excluded with zero
ok    contiguity: 39 ranges, rows 0 to 2535, no gaps or overlaps
ok    array: shape (2535, 1536), float16
ok    tile table: 2535 rows, each in its slide's range, tiles numbered from 0
ok    finite: all 3,893,760 values finite
ok    manifest: masks_used false; 39 slides, 2535 tiles
ok    fingerprint: sha256 f609a166c5d0... matches
ok    round trip: all 39 slides identical to their .pt files
features: memmap, float16, (2535, 1536)


,slide_id,start,end,n_tiles
0,0005f7aaab2800f6170c399693a96917,0,59,59
1,0704a3d9638d1657d8b36968ce2b92f8,59,118,59
2,0d11cd4b1285e9704ecd9fbee3ee8544,118,193,75
3,142aba6117eac17884a4bb4c99f7b78a,193,272,79
4,1b117dcc119ec9a258eccc88bd419b0f,272,326,54


,slide_id,tile,x,y,tissue_frac
0,0005f7aaab2800f6170c399693a96917,0,3584,3584,0.295918
1,0005f7aaab2800f6170c399693a96917,1,4480,3584,0.778061
2,0005f7aaab2800f6170c399693a96917,2,5376,3584,0.126594
3,0005f7aaab2800f6170c399693a96917,3,3584,4480,0.165179
4,0005f7aaab2800f6170c399693a96917,4,4480,4480,0.406569


{'slides': 39, 'tiles': 2535, 'excluded_slides': {'3790f55cad63053e956fb73027179707': 'empty file: every pixel white at every level, zero tiles (FINDINGS, Stage 10)'}, 'files': {'features': 'features.npy', 'index': 'index.parquet', 'tiles': 'tiles.parquet'}, 'row_order': 'slides sorted by slide_id; within a slide, the order of its .pt file', 'features_sha256': 'f609a166c5d0e654969cc09ddee3148d801f2ffa1662b705eeecad86f1733ee0', 'created': '2026-11-13T07:57:33+00:00', 'code_commit': 'b77b3d2'}


## Seeing it fail

A verification that has never been seen to fail proves nothing. Each fault
below is planted on a fresh scratch copy, one at a time, and the check that
should catch it must fail and say so.

| Fault | Check that must catch it |
|---|---|
| two slides' `start`/`end` swapped in the index (the silent-swap bug) | round trip, tile table |
| a gap between two slides' ranges | contiguity |
| one slide's index row deleted | slides |
| `masks_used` set to true | manifest |
| one NaN written into the array | finite, fingerprint |
| one value of the array changed | fingerprint, round trip |
| two tiles of one slide swapped | round trip |
| a `.part` file left in `slides/` | `check_complete` stops |
| a zero-tile slide not in `EXCLUDED` | `pack` stops |
| an excluded slide that has tiles | `pack` stops |
| packing again without `overwrite` | `pack` stops |

In [5]:
"""Plant each fault on a fresh copy and record what caught it.

1. The verify faults start from a copy of the packed clean set. Each plant
   function damages one thing: the index, the manifest or the array (opened
   read-write, written, flushed). Then verify runs, and the checks that failed
   must include the ones the table above names.
2. The pack faults start from an unpacked copy (or the packed one, for packing
   twice). The call must stop with an error.
3. One row per fault: what was planted, which checks failed or what error
   stopped it, and whether that is what had to happen. The cell stops if any
   fault was missed.
"""


def edit_index(features_dir, change):
    path = os.path.join(set_dir(VERSION, features_dir), INDEX_FILE)
    index = pd.read_parquet(path)
    change(index)
    index.to_parquet(path, index=False)


def edit_array(features_dir, change):
    array = np.load(os.path.join(set_dir(VERSION, features_dir), FEATURES_FILE), mmap_mode="r+")
    change(array)
    array.flush()
    del array


def swap_ranges(index):
    index.loc[[1, 2], ["start", "end"]] = index.loc[[2, 1], ["start", "end"]].to_numpy()


def gap(index):
    index.loc[3, "start"] += 1


def drop_row(index):
    index.drop(index=5, inplace=True)


def masks_true(features_dir):
    manifest = read_manifest(VERSION, features_dir)
    manifest["masks_used"] = True
    write_manifest(VERSION, manifest, features_dir)


def put_nan(array):
    array[10, 7] = np.nan


def change_value(array):
    array[10, 7] = array[10, 7] + np.float16(1)


def swap_tiles(array):
    array[[0, 1]] = array[[1, 0]]


verify_faults = [
    ("two slides' start/end swapped in the index", lambda d: edit_index(d, swap_ranges), {"round trip", "tile table"}),
    ("a gap between two slides' ranges", lambda d: edit_index(d, gap), {"contiguity"}),
    ("one slide's index row deleted", lambda d: edit_index(d, drop_row), {"slides"}),
    ("masks_used set to true", masks_true, {"manifest"}),
    ("one NaN written into the array", lambda d: edit_array(d, put_nan), {"finite", "fingerprint"}),
    ("one value of the array changed", lambda d: edit_array(d, change_value), {"fingerprint", "round trip"}),
    ("two tiles of one slide swapped", lambda d: edit_array(d, swap_tiles), {"round trip"}),
]

rows = []
for number, (fault, plant, must_catch) in enumerate(verify_faults):
    features_dir = make_scratch(f"verify-{number}", packed_from=clean)
    plant(features_dir)
    try:
        verify(VERSION, scratch_ids, features_dir)
        failed = []
    except VerificationError as error:
        failed = error.failed
    rows.append({"fault": fault, "must be caught by": ", ".join(sorted(must_catch)),
                 "caught by": ", ".join(failed) or "nothing", "ok": must_catch <= set(failed)})


def leftover_part(features_dir):
    with open(slide_file(VERSION, scratch_ids[0], features_dir) + PART_SUFFIX, "wb") as f:
        f.write(b"half a file")
    check_complete(VERSION, scratch_ids, features_dir)


def unlisted_zero_tile(features_dir):
    pack(VERSION, scratch_ids, {}, features_dir)


def excluded_has_tiles(features_dir):
    has_tiles = next(s for s in scratch_ids if s not in EXCLUDED)
    pack(VERSION, scratch_ids, {**EXCLUDED, has_tiles: "planted"}, features_dir)


def pack_twice(features_dir):
    pack(VERSION, scratch_ids, EXCLUDED, features_dir)


pack_faults = [
    ("a .part file left in slides/", leftover_part, None),
    ("a zero-tile slide not in EXCLUDED", unlisted_zero_tile, None),
    ("an excluded slide that has tiles", excluded_has_tiles, None),
    ("packing again without overwrite", pack_twice, clean),
]
for number, (fault, call, packed_from) in enumerate(pack_faults):
    features_dir = make_scratch(f"pack-{number}", packed_from=packed_from)
    try:
        call(features_dir)
        stopped = None
    except (ValueError, FileExistsError) as error:
        stopped = f"{type(error).__name__}: {str(error).splitlines()[-1].strip()}"
    rows.append({"fault": fault, "must be caught by": "stops", "caught by": stopped or "nothing",
                 "ok": stopped is not None})

faults = pd.DataFrame(rows)
with pd.option_context("display.max_colwidth", 120):
    display(faults)
if not faults["ok"].all():
    raise RuntimeError(f"{(~faults['ok']).sum()} planted faults were not caught")
print(f"all {len(faults)} planted faults caught")
shutil.rmtree(SCRATCH_DIR)

,fault,must be caught by,caught by,ok
0,two slides' start/end swapped in the index,"round trip, tile table","contiguity, tile table, round trip",True
1,a gap between two slides' ranges,contiguity,"contiguity, tile table, round trip",True
2,one slide's index row deleted,slides,"slides, contiguity, tile table, manifest",True
3,masks_used set to true,manifest,manifest,True
4,one NaN written into the array,"fingerprint, finite","finite, fingerprint, round trip",True
5,one value of the array changed,"fingerprint, round trip","fingerprint, round trip",True
6,two tiles of one slide swapped,round trip,"fingerprint, round trip",True
7,a .part file left in slides/,stops,ValueError: 1 other files in slides/: 0005f7aaab2800f6170c399693a96917.pt.part,True
8,a zero-tile slide not in EXCLUDED,stops,ValueError: zero-tile slides not in the exclusions: 3790f55cad63053e956fb73027179707,True
9,an excluded slide that has tiles,stops,ValueError: excluded slides that have tiles: 0005f7aaab2800f6170c399693a96917,True


all 11 planted faults caught


## The real set: is extraction complete?

Packing a partial set produces something that looks finished. So before
anything is written: the slides folder must hold exactly the expected 10,616
slides, with no leftovers from a crashed write, and the manifest must name this
version and say `masks_used: false`.

In [6]:
"""check_complete on the real set: stops with every problem listed, or passes silently."""

check_complete(VERSION, expected_ids)
print(f"{VERSION}: all {len(expected_ids)} expected slides present, nothing else in {SLIDES_SUBDIR}/, "
      f"manifest says masks_used: false")

uni2h-L1-224-t144-hsv-v1: all 10616 expected slides present, nothing else in slides/, manifest says masks_used: false


## How many tiles?

The total sets the size of the array, which is created at full size before
any slide is written. The only zero-tile slides found must be exactly the ones
in `EXCLUDED`.

In [7]:
"""Count every slide's tiles (each file read and its format checked), and compare the zero-tile slides with EXCLUDED."""

counts = count_tiles(VERSION, expected_ids, progress=tqdm)
zero = sorted(counts[counts == 0].index)
print(f"{counts.sum():,} tiles over {len(counts):,} slides; per slide median {counts.median():.0f}, "
      f"min {counts.min()}, max {counts.max()}")
print(f"slides at the 144-tile cap: {(counts == 144).sum():,}")
print(f"zero-tile slides: {zero}")
if set(zero) != set(EXCLUDED):
    raise ValueError(f"zero-tile slides {zero} are not exactly the exclusions {sorted(EXCLUDED)}")

691,236 tiles over 10,616 slides; per slide median 66, min 0, max 144
slides at the 144-tile cap: 70
zero-tile slides: ['3790f55cad63053e956fb73027179707']


## Pack

One slide at a time in sorted slide-ID order, so memory stays flat. All three
files are written under temporary names and renamed only when all three are
complete, so a crash never leaves a packed set that looks finished. Then
`features.npy` is fingerprinted and the manifest gains its `packed` section,
keeping the extraction's own commit alongside the packing commit.

In [8]:
"""Pack the real set, then show the summary and the packed section of the manifest."""

summary = pack(VERSION, expected_ids, EXCLUDED, overwrite=OVERWRITE, progress=tqdm)
print(f"packed {summary['slides']:,} slides, {summary['tiles']:,} tiles in {summary['seconds']:.0f} s")
for name, size in summary["bytes"].items():
    print(f"  {name:15s} {size / 1e6:9.1f} MB")
print(f"excluded: {summary['excluded']}")
print(read_manifest(VERSION)["packed"])

packed 10,615 slides, 691,236 tiles in 9 s
  features.npy       2123.5 MB
  index.parquet         0.5 MB
  tiles.parquet         2.5 MB
excluded: {'3790f55cad63053e956fb73027179707': 'empty file: every pixel white at every level, zero tiles (FINDINGS, Stage 10)'}
{'slides': 10615, 'tiles': 691236, 'excluded_slides': {'3790f55cad63053e956fb73027179707': 'empty file: every pixel white at every level, zero tiles (FINDINGS, Stage 10)'}, 'files': {'features': 'features.npy', 'index': 'index.parquet', 'tiles': 'tiles.parquet'}, 'row_order': 'slides sorted by slide_id; within a slide, the order of its .pt file', 'features_sha256': '14386ff532415f186d9ce7f7ac8a7cb66dfdffa351e38ec0063d9a5e4bdc41dd', 'created': '2026-11-13T07:57:35+00:00', 'code_commit': 'b77b3d2'}


## Verify

Every check runs and every failure is collected before the verification
fails, so one run shows everything wrong:

1. every expected slide is in the index once, or listed as excluded;
2. no packed slide has zero tiles, and every excluded one does;
3. the ranges are contiguous: start at 0, no gaps, no overlaps, end at the
   last row;
4. the array's shape and dtype match the manifest;
5. the tile table has one row per array row, in the same order;
6. no NaN or infinity, scanned in chunks;
7. `masks_used` is false and the manifest's counts match the files;
8. the fingerprint of `features.npy` matches the manifest;
9. round trip: every slide's packed rows equal its `.pt` file exactly.

In [9]:
"""Verify the real set, round trip included: every slide compared with its .pt file."""

started = time.perf_counter()
report = verify(VERSION, expected_ids, progress=tqdm)
print("\n".join(report))
print(f"verified in {time.perf_counter() - started:.0f} s")

ok    slides: 10615 packed + 1 excluded = 10616 expected
ok    no empty: smallest slide has 4 tiles; 1 excluded with zero
ok    contiguity: 10615 ranges, rows 0 to 691236, no gaps or overlaps
ok    array: shape (691236, 1536), float16
ok    tile table: 691236 rows, each in its slide's range, tiles numbered from 0
ok    finite: all 1,061,738,496 values finite
ok    manifest: masks_used false; 10615 slides, 691236 tiles
ok    fingerprint: sha256 14386ff53241... matches
ok    round trip: all 10615 slides identical to their .pt files
verified in 6 s


## Is it faster?

Something to look at, not a test: the time to read every slide's features
once from the 10,616 `.pt` files, against once from the packed set through
the index.

In [10]:
"""Time reading every slide's features once, from the .pt files and from the packed set.

1. The slides are read in one shuffled order (seed 0), as a training epoch
   would, and the same order is used for both.
2. Cold: before each timing, every file involved is dropped from the
   operating system's page cache (posix_fadvise DONTNEED, Linux only), so the
   reads come from the disk. Warm: the same reads again straight after, from
   memory. Training sees something in between.
3. From the packed set each slide's rows are copied out of the memory map
   (np.array), which is what forces the read.
"""

packed = open_packed(VERSION)
order = np.random.default_rng(0).permutation(len(packed.index))
ids = packed.index["slide_id"].to_numpy()
pt_paths = [slide_file(VERSION, slide_id) for slide_id in ids]


def evict(paths):
    if not hasattr(os, "posix_fadvise"):
        return False
    for path in paths:
        fd = os.open(path, os.O_RDONLY)
        try:
            os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED)
        finally:
            os.close(fd)
    return True


def read_pt():
    for i in order:
        torch.load(pt_paths[i], weights_only=True)["features"]


def read_packed():
    for i in order:
        np.array(packed.features[packed.starts[i]:packed.ends[i]])


times = {}
for name, read, paths in [(".pt files", read_pt, pt_paths),
                          ("packed", read_packed, [os.path.join(set_dir(VERSION), FEATURES_FILE)])]:
    cold = evict(paths)
    for label in ["cold" if cold else "first", "warm"]:
        started = time.perf_counter()
        read()
        times[(name, label)] = time.perf_counter() - started
for (name, label), seconds in times.items():
    print(f"{name:10s} {label:5s} {seconds:7.1f} s   {seconds / len(order) * 1e3:6.2f} ms per slide")

.pt files  cold      6.1 s     0.58 ms per slide
.pt files  warm      1.9 s     0.18 ms per slide
packed     cold      1.0 s     0.10 ms per slide
packed     warm      0.1 s     0.01 ms per slide


## Result

Run 2026-11-13 on the RTX 4060 PC (14 GB RAM, NVMe disk).

- **Version:** `uni2h-L1-224-t144-hsv-v1`, the tissue set from notebook 9
  (extraction commit `b778b50`, kept in the manifest).
- **Complete:** all 10,616 expected slides have a per-slide file, with nothing
  else in `slides/`; the manifest says `masks_used: false`.
- **Packed:** 10,615 slides, 691,236 tiles (median 66 per slide, 4 to 144).
  Left out by name: `3790f55cad63053e956fb73027179707`, the empty all-white
  file with zero tiles (FINDINGS, Stage 10). It is the only zero-tile slide.
- **Files:** `features.npy` 2,123.5 MB (fp16 [691,236, 1536]),
  `index.parquet` 0.5 MB, `tiles.parquet` 2.5 MB. Packing took 9 s with the
  per-slide files already in memory.
- **Verification:** all nine checks pass, round trip over every slide
  included, in 6 s. On the scratch copy all 11 planted faults were caught.
- **Read speed** (every slide once, shuffled): from disk 6.1 s from the `.pt`
  files against 1.0 s packed; from memory 1.9 s against 0.1 s.
- **Commit:** packed at `b77b3d2` (the manifest's `packed.code_commit`).

Train from the packed set through `open_packed`. `tests/test_feature_set.py`
re-runs this verification at any time.